# Laboratorio #5 – Análisis de tráfico de red
**CC3094 – Security Data Science | Semestre I - 2026**

Universidad del Valle de Guatemala

In [1]:
%pip install scapy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from scapy.all import * 

import pandas as pd
import numpy as np
import binascii
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest

sns.set(color_codes=True)
%matplotlib inline

---
# Parte 1 - Análisis de paquetes

## Testeo de la herramienta

### Inciso 1 – Captura de 10 paquetes con Scapy

In [4]:
num_of_packets_to_sniff = 10
pcap_10 = sniff(count=num_of_packets_to_sniff)

# rdpcap returns packet list
## packetlist object can be enumerated
print(type(pcap_10))
print(len(pcap_10))
print(pcap_10)
pcap_10[0]

<class 'scapy.plist.PacketList'>
10
<Sniffed: TCP:7 UDP:3 ICMP:0 Other:0>


<Ether  dst=01:00:5e:00:00:fb src=bc:d7:d4:f8:af:c0 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=90 id=35747 flags=DF frag=0 ttl=255 proto=udp chksum=0x4e49 src=192.168.0.2 dst=224.0.0.251 |<UDP  sport=5353 dport=5353 len=70 chksum=0x6ffd |<DNS  id=0 qr=0 opcode=QUERY aa=0 tc=0 rd=0 ra=0 z=0 ad=0 cd=0 rcode=ok qdcount=1 ancount=0 nscount=0 arcount=1 qd=[<DNSQR  qname=b'lb._dns-sd._udp.local.' qtype=PTR unicastresponse=0 qclass=IN |>] ar=[<DNSRROPT  rrname=b'.' type=OPT rclass=1440 extrcode=0 version=0 z=4500 rdata=[<EDNS0OWN  optcode=Owner optlen=8 v=0 s=0 primary_mac=bc:d7:d4:f8:af:c0 |>] |>] |>>>>

### Inciso 2 – Mostrar Src Address, Dst Address, Src Port y Dst Port

In [5]:
print(f"{'#':<4} {'Src Address':<18} {'Dst Address':<18} {'Src Port':<12} {'Dst Port':<12}")
print("-" * 70)

for i, pkt in enumerate(pcap_10):
    if IP in pkt:
        src_ip  = pkt[IP].src
        dst_ip  = pkt[IP].dst
        if TCP in pkt:
            sport, dport = pkt[TCP].sport, pkt[TCP].dport
        elif UDP in pkt:
            sport, dport = pkt[UDP].sport, pkt[UDP].dport
        else:
            sport, dport = 'N/A', 'N/A'
        print(f"{i:<4} {src_ip:<18} {dst_ip:<18} {str(sport):<12} {str(dport):<12}")

#    Src Address        Dst Address        Src Port     Dst Port    
----------------------------------------------------------------------
0    192.168.0.2        224.0.0.251        5353         5353        
1    192.168.0.2        224.0.0.251        5353         5353        
4    192.168.0.23       51.116.246.106     62653        443         
5    192.168.0.23       51.116.246.106     62653        443         
6    192.168.0.23       51.116.246.106     62653        443         
7    192.168.0.23       51.116.246.106     62653        443         
8    192.168.0.23       51.116.246.106     62653        443         
9    192.168.0.23       51.116.246.106     62653        443         


---
## Estadísticas y detección con Z-Score

### Inciso 1 – Cargar el archivo analisis_paquetes.pcap

In [ ]:
pcap = rdpcap('analisis_paquetes.pcap')
print(f"PCAP cargado correctamente. Total de paquetes: {len(pcap)}")

### Inciso 2 – Convertir el PCAP a un DataFrame

In [ ]:
rows = []

for pkt in pcap:
    if IP not in pkt:
        continue

    src_ip  = pkt[IP].src
    dst_ip  = pkt[IP].dst
    pkt_len = pkt[IP].len
    proto   = pkt[IP].proto
    ts      = float(pkt.time)

    if TCP in pkt:
        sport    = pkt[TCP].sport
        dport    = pkt[TCP].dport
        raw      = bytes(pkt[TCP].payload)
    elif UDP in pkt:
        sport    = pkt[UDP].sport
        dport    = pkt[UDP].dport
        raw      = bytes(pkt[UDP].payload)
    else:
        sport, dport = None, None
        raw = b''

    payload_len = len(raw)
    payload_hex = binascii.hexlify(raw)

    rows.append({
        'src':         src_ip,
        'dst':         dst_ip,
        'sport':       sport,
        'dport':       dport,
        'pkt_len':     pkt_len,
        'proto':       proto,
        'time':        ts,
        'payload':     payload_len,
        'payload_raw': raw,
        'payload_hex': payload_hex,
    })

df = pd.DataFrame(rows)
print(f"DataFrame creado con {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

In [ ]:
print("Shape:", df.shape)
df.dtypes

### Inciso 3 – Estadísticas básicas

In [ ]:
# a. IP origen más frecuente
src_desc = df['src'].describe()
top_src = src_desc['top']
print("=== a. IP Origen más frecuente ===")
print(src_desc)
print(f"\n► IP origen más frecuente: {top_src} ({src_desc['freq']} paquetes)")

print()

# b. IP destino más frecuente
dst_desc = df['dst'].describe()
top_dst = dst_desc['top']
print("=== b. IP Destino más frecuente ===")
print(dst_desc)
print(f"\n► IP destino más frecuente: {top_dst} ({dst_desc['freq']} paquetes)")

In [ ]:
# c. ¿A qué IPs se comunica la IP origen más frecuente?
print(f"=== c. IPs con las que se comunica {top_src} ===")
ips_con_top_src = df[df['src'] == top_src]['dst'].unique()
print(ips_con_top_src)

In [ ]:
# d. ¿A qué puertos destino se comunica la IP origen más frecuente?
print(f"=== d. Puertos destino a los que se comunica {top_src} ===")
dports_top_src = df[df['src'] == top_src]['dport'].value_counts()
print(dports_top_src)

In [ ]:
# e. ¿A qué puertos origen se comunica la IP destino más frecuente?
print(f"=== e. Puertos origen desde los que envía {top_dst} ===")
sports_top_dst = df[df['src'] == top_dst]['sport'].value_counts()
print(sports_top_dst)

In [ ]:
# f. Propósito de los puertos que más aparecen
print("=== f. Propósito de los puertos más frecuentes ===")
print("""
Puerto 53 – DNS (Domain Name System)
  Es el puerto estándar utilizado para la resolución de nombres de dominio.
  Opera principalmente sobre UDP (aunque también puede usar TCP para
  respuestas grandes o transferencias de zona).
  
  El puerto 53 es prácticamente el ÚNICO puerto presente en toda la captura,
  tanto en origen como en destino. Esto indica que todo el tráfico capturado
  corresponde a consultas y respuestas DNS.
"""
)

### Inciso 4 – Z-Score sobre el tamaño de payload

In [ ]:
# a. Z-score calculado con media y desviación estándar del propio pcap
mu_pcap    = df['payload'].mean()
sigma_pcap = df['payload'].std()

df['zscore_pcap'] = (df['payload'] - mu_pcap) / sigma_pcap

anomalias_2 = df[df['zscore_pcap'].abs() > 2]
anomalias_3 = df[df['zscore_pcap'].abs() > 3]

print(f"Media del payload: {mu_pcap:.2f} bytes")
print(f"Desviación estándar: {sigma_pcap:.2f} bytes")
print()
print(f"Paquetes con |Z| > 2: {len(anomalias_2)}")
print(f"Paquetes con |Z| > 3: {len(anomalias_3)}")
print()

if len(anomalias_3) > 0:
    print("Paquetes anómalos (|Z| > 3):")
    print(anomalias_3[['src', 'dst', 'sport', 'dport', 'payload', 'zscore_pcap']].to_string())
else:
    print("No se detectaron paquetes anómalos con |Z| > 3 (usando estadísticas del pcap).")
    print()
    if len(anomalias_2) > 0:
        print("Paquetes con |Z| > 2:")
        print(anomalias_2[['src', 'dst', 'sport', 'dport', 'payload', 'zscore_pcap']].to_string())

In [ ]:
# b. Histograma de payload y análisis de distribución
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma
ax = axes[0]
data = df['payload'].values
ax.hist(data, bins=20, density=True, alpha=0.65, color='#4A90D9', edgecolor='white', label='Distribución empírica')
mu, sigma = data.mean(), data.std()
x = np.linspace(data.min(), data.max(), 300)
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'Gaussiana ajustada\nN({mu:.0f}, {sigma:.0f}²)')
ax.set_title('Histograma de Payload vs. Ajuste Gaussiano', fontweight='bold')
ax.set_xlabel('Payload (bytes)')
ax.set_ylabel('Densidad')
ax.legend()

# Boxplot
ax2 = axes[1]
ax2.boxplot(data, vert=False, patch_artist=True,
            boxprops=dict(facecolor='#4A90D9', alpha=0.6),
            medianprops=dict(color='red', linewidth=2))
ax2.set_title('Boxplot de Payload', fontweight='bold')
ax2.set_xlabel('Payload (bytes)')
ax2.set_yticks([])

plt.tight_layout()
plt.show()

print("""
Análisis de la distribución:
La distribución del payload NO es normal (no gaussiana). El histograma muestra
una distribución bimodal/dispersa: muchos paquetes pequeños (consultas DNS
legítimas, ~37 bytes) y muchos paquetes grandes (respuestas con datos
exfiltrados, 900-1023 bytes). Esta distribución es muy diferente a una
campana de Gauss simétrica.

Esto explica por qué el Z-score basado en el pcap no detecta anomalías:
la media se desplaza hacia el centro entre los dos grupos (~481 bytes),
y la desviación estándar es tan grande que ningún punto queda a más
de 3σ de la media inflada. El Z-score "normal" pierde sensibilidad
cuando los datos anómalos son tantos que distorsionan los parámetros.
""")

In [ ]:
# c. Z-score con conocimiento de dominio (valores típicos de DNS)
mu_dns    = 50    # tamaño típico de consulta DNS en bytes
sigma_dns = 15    # desviación estándar típica

df['zscore_domain'] = (df['payload'] - mu_dns) / sigma_dns

anomalias_domain_2 = df[df['zscore_domain'].abs() > 2]
anomalias_domain_3 = df[df['zscore_domain'].abs() > 3]

print(f"Parámetros de referencia DNS: mu={mu_dns} bytes, sigma={sigma_dns} bytes")
print()
print(f"Paquetes anómalos con |Z_domain| > 2: {len(anomalias_domain_2)}")
print(f"Paquetes anómalos con |Z_domain| > 3: {len(anomalias_domain_3)}")
print()
print("Muestra de paquetes anómalos (|Z| > 3, primeros 10):")
print(anomalias_domain_3[['src', 'dst', 'sport', 'dport', 'payload', 'zscore_domain']].head(10).to_string())

In [ ]:
# d. Lección sobre el conocimiento de dominio
print("""
=== d. Importancia del conocimiento de dominio en la detección de anomalías ===

Este experimento demuestra que las técnicas de detección de anomalías son
tan buenas como la referencia con la que trabajan.

Cuando calculamos el Z-score usando la media y desviación del propio pcap,
el tráfico malicioso "contamina" los parámetros estadísticos: la media sube
artificialmente y la desviación aumenta, haciendo que el modelo considere
los paquetes grandes como "normales dentro del pcap".

En cambio, al usar el conocimiento de dominio (mu=50 bytes, sigma=15 bytes
para DNS), tenemos una línea base del comportamiento ESPERADO del protocolo.
Cualquier desviación significativa de esa expectativa se detecta como anomalía.

Lección clave: Un analista de seguridad debe conocer el protocolo que analiza.
El Z-score ciego a los datos es susceptible de ser engañado cuando el ataque
es persistente y voluminoso. El conocimiento del protocolo actúa como ancla
de referencia robusta ante la contaminación de los datos.
""")

---
## Gráficas de payload

### Inciso 5 – Visualizaciones 2D de payload

In [ ]:
# a. Payload enviado por IP origen
fig, ax = plt.subplots(figsize=(10, 4))
src_payload = df.groupby('src')['payload'].sum().sort_values()
colors_src = ['#E74C3C' if ip == top_src else '#4A90D9' for ip in src_payload.index]
bars = ax.barh(src_payload.index, src_payload.values, color=colors_src, edgecolor='white')
for bar, val in zip(bars, src_payload.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Suma de payload (bytes)')
ax.set_ylabel('IP Origen')
ax.set_title('a. Payload total enviado por IP Origen', fontweight='bold')
red_patch = mpatches.Patch(color='#E74C3C', label=f'IP más frecuente ({top_src})')
blue_patch = mpatches.Patch(color='#4A90D9', label='Otras IPs')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.show()

In [ ]:
# b. Payload recibido por IP destino
fig, ax = plt.subplots(figsize=(10, 4))
dst_payload = df.groupby('dst')['payload'].sum().sort_values()
colors_dst = ['#E74C3C' if ip == top_dst else '#2ECC71' for ip in dst_payload.index]
bars = ax.barh(dst_payload.index, dst_payload.values, color=colors_dst, edgecolor='white')
for bar, val in zip(bars, dst_payload.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Suma de payload (bytes)')
ax.set_ylabel('IP Destino')
ax.set_title('b. Payload total recibido por IP Destino', fontweight='bold')
red_patch = mpatches.Patch(color='#E74C3C', label=f'IP más frecuente ({top_dst})')
green_patch = mpatches.Patch(color='#2ECC71', label='Otras IPs')
ax.legend(handles=[red_patch, green_patch])
plt.tight_layout()
plt.show()

In [ ]:
# c. Payload enviado por puerto origen
fig, ax = plt.subplots(figsize=(10, 5))
sport_payload = df.groupby('sport')['payload'].sum().sort_values().tail(15)
ax.barh(sport_payload.index.astype(str), sport_payload.values, color='#9B59B6', edgecolor='white')
for i, val in enumerate(sport_payload.values):
    ax.text(val + 100, i, f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Suma de payload (bytes)')
ax.set_ylabel('Puerto Origen')
ax.set_title('c. Payload total enviado por Puerto Origen (top 15)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# d. Payload recibido por puerto destino
fig, ax = plt.subplots(figsize=(10, 5))
dport_payload = df.groupby('dport')['payload'].sum().sort_values().tail(15)
ax.barh(dport_payload.index.astype(str), dport_payload.values, color='#E67E22', edgecolor='white')
for i, val in enumerate(dport_payload.values):
    ax.text(val + 100, i, f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Suma de payload (bytes)')
ax.set_ylabel('Puerto Destino')
ax.set_title('d. Payload total recibido por Puerto Destino (top 15)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Detección automática con Isolation Forest

### Inciso 6 – Isolation Forest

In [ ]:
# a. Entrenar Isolation Forest
# Número de paquetes anómalos detectados en el inciso 4c (|Z_domain| > 3)
n_anomalias_4c = len(anomalias_domain_3)
n_total        = len(df)
contamination  = n_anomalias_4c / n_total

print(f"Paquetes anómalos detectados en 4c: {n_anomalias_4c}")
print(f"Total de paquetes: {n_total}")
print(f"Valor de contamination: {contamination:.4f} ({contamination*100:.2f}%)")
print()

# Features numéricas
features_if = ['payload', 'pkt_len']
X = df[features_if].values

# Entrenar el modelo
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42
)
iso_forest.fit(X)

# Predicción: -1 = anomalía, 1 = normal
df['if_prediction'] = iso_forest.predict(X)
df['if_score']      = iso_forest.score_samples(X)   # menor score = más anómalo
df['if_anomaly']    = df['if_prediction'] == -1

if_anomalias = df[df['if_anomaly']]
print(f"Paquetes marcados como anomalías por Isolation Forest: {len(if_anomalias)}")
print()
print("Detalle de las anomalías detectadas:")
print(if_anomalias[['src', 'dst', 'sport', 'dport', 'payload', 'pkt_len', 'if_score']].to_string())

In [ ]:
# Visualización del Isolation Forest
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: payload vs pkt_len coloreado por predicción
ax = axes[0]
normal = df[~df['if_anomaly']]
anoms  = df[df['if_anomaly']]
ax.scatter(normal['payload'], normal['pkt_len'], c='#3498DB', alpha=0.6, s=50, label='Normal', edgecolors='white')
ax.scatter(anoms['payload'],  anoms['pkt_len'],  c='#E74C3C', alpha=0.9, s=80, label='Anomalía (IF)', marker='X', edgecolors='black')
ax.set_xlabel('Payload (bytes)')
ax.set_ylabel('Longitud del paquete IP (bytes)')
ax.set_title('Isolation Forest: payload vs. pkt_len', fontweight='bold')
ax.legend()

# Plot 2: anomaly score por índice de paquete
ax2 = axes[1]
ax2.plot(df.index, df['if_score'], color='#95A5A6', alpha=0.7, linewidth=1, label='Anomaly score')
threshold_score = df[df['if_anomaly']]['if_score'].max()
ax2.axhline(y=threshold_score, color='#E74C3C', linestyle='--', linewidth=1.5, label=f'Umbral (score={threshold_score:.3f})')
ax2.scatter(anoms.index, anoms['if_score'], c='#E74C3C', s=60, zorder=5, label='Anomalías')
ax2.set_xlabel('Índice de paquete')
ax2.set_ylabel('Anomaly score (menor = más anómalo)')
ax2.set_title('Anomaly Score por paquete', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# b. ¿Las anomalías coinciden con las detectadas por Z-score?
print("=== b. Comparación Isolation Forest vs. Z-Score con conocimiento de dominio ===")
print()

# IPs anómalas según Z-score
ips_zscore = set(anomalias_domain_3['src'].unique()) | set(anomalias_domain_3['dst'].unique())
# IPs anómalas según Isolation Forest
ips_if     = set(if_anomalias['src'].unique()) | set(if_anomalias['dst'].unique())

print(f"IPs involucradas según Z-score (dominio): {ips_zscore}")
print(f"IPs involucradas según Isolation Forest:  {ips_if}")
print(f"Intersección: {ips_zscore & ips_if}")
print()
print("""
Análisis:
Ambas técnicas convergen en identificar el mismo par de IPs sospechosas.
El Isolation Forest detecta las anomalías puramente por la forma del dato
(payload y longitud de paquete inusualmente grandes) sin ningún conocimiento
previo del protocolo.

Los paquetes anómalos de ambas técnicas corresponden a los mismos paquetes
con payloads muy superiores al rango normal de DNS. Esto refuerza la
sospecha de actividad maliciosa y valida el uso complementario de ambos
métodos en un flujo de trabajo SOC.
""")

---
## Investigación del payload (confirmación manual)

Las técnicas anteriores señalaron tráfico sospechoso. Ahora investigamos el contenido para confirmar el tipo de ataque.

### Inciso 7 – Investigación del payload

In [ ]:
# a. DataFrame con conexiones de la IP origen más frecuente
df_top_src = df[df['src'] == top_src].copy()
print(f"=== a. Conexiones originadas por {top_src} ===")
print(f"Total de paquetes: {len(df_top_src)}")
print()
print(df_top_src[['src', 'dst', 'sport', 'dport', 'payload', 'pkt_len']].to_string())

In [ ]:
# b. DataFrame con Src Address, Dst Address agrupadas por payload
print(f"=== b. Src/Dst agrupadas por suma de payload ===")
df_grouped = df_top_src.groupby(['src', 'dst'])['payload'].sum().reset_index()
df_grouped = df_grouped.sort_values('payload', ascending=False)
print(df_grouped.to_string(index=False))

In [ ]:
# c. IP que más ha intercambiado bytes con la IP más frecuente
print(f"=== c. IP que más ha intercambiado bytes con {top_src} ===")

# Calcular intercambio bidireccional total
exchange = df.groupby(['src', 'dst'])['payload'].sum().reset_index()

# Filtrar conexiones que involucran a top_src
exchange_top = exchange[(exchange['src'] == top_src) | (exchange['dst'] == top_src)].copy()
exchange_top['other_ip'] = exchange_top.apply(
    lambda r: r['dst'] if r['src'] == top_src else r['src'], axis=1
)
total_by_ip = exchange_top.groupby('other_ip')['payload'].sum().sort_values(ascending=False)

suspicious_ip = total_by_ip.index[0]
print(f"IP con mayor intercambio de bytes: {suspicious_ip}")
print(f"Total de bytes intercambiados: {total_by_ip[suspicious_ip]:,}")
print()
print("Resumen de intercambio:")
print(total_by_ip)

In [ ]:
# d. DataFrame con la conversación entre la IP más frecuente y la IP sospechosa
print(f"=== d. Conversación entre {top_src} y {suspicious_ip} ===")

df_conversation = df[
    ((df['src'] == top_src)    & (df['dst'] == suspicious_ip)) |
    ((df['src'] == suspicious_ip) & (df['dst'] == top_src))
].copy()

print(f"Total de paquetes en la conversación: {len(df_conversation)}")
print()
print(df_conversation[['src', 'dst', 'sport', 'dport', 'payload', 'pkt_len']].to_string())

In [ ]:
# e. Obtener los payloads y añadirlos a un array
payloads_array = df_conversation['payload_raw'].tolist()

print(f"=== e. Array de payloads ===")
print(f"Número de elementos en el array: {len(payloads_array)}")
print(f"Tamaños de cada payload: {[len(p) for p in payloads_array]}")

In [ ]:
# f. Mostrar el contenido del primer array
print("=== f. Contenido del primer payload ===")
primer_payload = payloads_array[0]
print(f"Longitud: {len(primer_payload)} bytes")
print()
print("Primeros 64 bytes en hex:")
print(primer_payload[:64].hex())
print()
print("Primeros 64 bytes como ASCII (errores reemplazados):")
print(primer_payload[:64].decode('utf-8', errors='replace'))
print()
print("Representación hexdump manual (primeros 128 bytes):")
chunk = primer_payload[:128]
for i in range(0, len(chunk), 16):
    hex_part   = ' '.join(f'{b:02x}' for b in chunk[i:i+16])
    ascii_part = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk[i:i+16])
    print(f"  {i:04x}  {hex_part:<48}  {ascii_part}")

In [ ]:
# g. Análisis de los primeros bytes – detección del ataque
print("=== g. Análisis forense del payload ===\n")

# Buscar la firma PNG en todos los payloads
PNG_MAGIC = bytes([0x89, 0x50, 0x4E, 0x47, 0x0D, 0x0A, 0x1A, 0x0A])  # \x89PNG\r\n\x1a\n

for idx, raw in enumerate(payloads_array):
    if PNG_MAGIC in raw:
        pos = raw.index(PNG_MAGIC)
        print(f"Payload #{idx}: firma PNG (magic bytes) encontrada en byte offset {pos}")
        print(f"  Hex magic: {PNG_MAGIC.hex()}")
        print(f"  Contexto: ...{raw[max(0,pos-4):pos+12].hex()}...")

print()

# Intentar reconstruir el archivo desde los payloads
reconstructed = b''
for raw in payloads_array:
    if PNG_MAGIC in raw:
        pos = raw.index(PNG_MAGIC)
        reconstructed += raw[pos:]
    elif len(raw) > 20:
        reconstructed += raw[20:]   # saltar encabezado DNS (~20 bytes)

print(f"Datos binarios reconstruidos: {len(reconstructed)} bytes")
print()

print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    ATAQUE DETECTADO: DNS TUNNELING                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  ¿Qué encontramos?                                                   ║
║  Los payloads de las consultas DNS al puerto 53 contienen los magic  ║
║  bytes de un archivo PNG (\x89PNG\r\n\x1a\n). Esto significa que    ║
║  un archivo de imagen binario está siendo transportado DENTRO de     ║
║  paquetes DNS, lo cual NO es el comportamiento normal de este        ║
║  protocolo.                                                          ║
║                                                                      ║
║  ¿En qué consiste el ataque?                                         ║
║  DNS Tunneling: técnica de exfiltración de datos que encapsula       ║
║  información arbitraria (archivos, comandos C2, credenciales) dentro ║
║  de consultas/respuestas DNS. El atacante aprovecha que el puerto 53 ║
║  generalmente no es bloqueado por firewalls y que el tráfico DNS es  ║
║  considerado "legítimo" en la mayoría de las redes.                  ║
║                                                                      ║
║  ¿Por qué es importante combinar técnicas + análisis manual?         ║
║  - El Z-score detectó paquetes estadísticamente inusuales.           ║
║  - El Isolation Forest confirmó las mismas anomalías sin parámetros. ║
║  - Pero SÓLO el análisis del payload confirma QUÉ se está            ║
║    transfiriendo (datos binarios de un archivo PNG).                 ║
║  - Sin inspección manual, podríamos saber que "algo raro pasa"      ║
║    pero no podríamos clasificar el ataque, estimar el impacto        ║
║    ni tomar acciones de respuesta adecuadas.                         ║
╚══════════════════════════════════════════════════════════════════════╝
""")